# Agent 第 18 课：观测（Observability）

上线前你写的都是 happy path；上线后问题永远在 happy path 之外。观测就是**出事时你还能 debug**的能力。

Phase 1 你把 thought / action / observation 直接 print 到终端，那只是在本地能用。上了托管 Bedrock Agent，你 print 不到模型 —— 得靠 AWS 这一套。

学完这节你能回答：
1. 一次 agent 调用里有多少"可观测事件"
2. CloudWatch Logs / Metrics / X-Ray / Model Invocation Logging 各负责什么
3. 什么 4 个核心指标必须盯
4. 出 bug 时从哪个日志开始看

## 1. 一次 invoke_agent 到底发生了什么

用户发一句话 → agent 内部一般是这样走的：

```
用户 input
  └─ [guardrail: input 扫描]
       └─ orchestration LLM 决策（"我要调 X 工具"）
            └─ [Action Group 的 Lambda]
                 └─ 你的业务代码（DB / API / …）
            └─ 或 [Knowledge Base retrieve]
       └─ orchestration LLM 再判断（"还需要调 Y 吗？"）
       └─ ……循环 N 次……
       └─ 生成最终答复
  └─ [guardrail: output 扫描]
用户 output
```

**每个箭头都可能挂**。观测就是让你知道是哪个箭头挂了、为什么挂。

## 2. 四个观测通道

| 通道 | 覆盖什么 | 默认开吗 | 主要用途 |
|---|---|---|---|
| **invoke_agent 的 trace** | orchestration、rationale、tool 调用、KB 调用、guardrail 命中 | API 调用时 `enableTrace=True` | 单次请求级 debug |
| **Model Invocation Logging** | 每次底层模型调用的 prompt / response（含 agent 内部的） | **默认关**，要开 | 看模型到底收到 / 吐出了什么 |
| **CloudWatch Logs（Lambda）** | Action Group Lambda 的 stdout / stderr | Lambda 默认开 | 业务代码 debug |
| **CloudWatch Metrics** | 调用数、延迟、错误率、token 数 | 自动 | 聚合看大盘 |
| **X-Ray traces** | 跨服务分布式链路（Agent → Lambda → Dynamo） | 需在 Lambda 和 Agent 侧开 | 性能瓶颈 / 跨服务 |

**组合使用**：metrics 发现异常 → trace 定位是哪次 → model invocation logs 看模型输入输出 → Lambda logs 看业务代码。

## 3. Trace：先从这里开始

第 13 课已经演示过 `enableTrace=True`。观测的第一块是把 trace **结构化**出来。一次调用的 trace 会按类型划分：

In [ ]:
import boto3, os, json, uuid
os.environ.setdefault("AWS_REGION", "us-west-2")

runtime = boto3.client("bedrock-agent-runtime")
logs    = boto3.client("logs")
cw      = boto3.client("cloudwatch")
bedrock = boto3.client("bedrock")

AGENT_ID       = "XXXXXXXXXX"
AGENT_ALIAS_ID = "YYYYYYYYYY"

In [ ]:
def collect_trace(text):
    sid = str(uuid.uuid4())
    stream = runtime.invoke_agent(
        agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID,
        sessionId=sid, inputText=text, enableTrace=True,
    )
    answer = []
    trace_buckets = {
        "preProcessing":     [],   # 改写用户输入（如有）
        "orchestration":     [],   # 核心 ReAct 循环
        "postProcessing":    [],   # 改写最终回答（如有）
        "guardrail":         [],
        "failure":           [],   # 出错
    }
    for ev in stream["completion"]:
        if "chunk" in ev:
            answer.append(ev["chunk"]["bytes"].decode())
        elif "trace" in ev:
            t = ev["trace"]["trace"]
            for k in trace_buckets:
                if k in t:
                    trace_buckets[k].append(t[k])
    return "".join(answer), trace_buckets

# answer, buckets = collect_trace("Beijing 现在温度多少？")
# print(answer)
# print("orchestration steps =", len(buckets["orchestration"]))

### orchestration trace 里关键字段

每一步 orchestration 通常有：
- `modelInvocationInput` —— AWS 拼给底层模型的完整 prompt
- `modelInvocationOutput` —— 模型原始回复（rationale / tool 决策）
- `rationale` —— agent 自己的思考（对应 Phase 1 的 "Thought"）
- `invocationInput` —— 决定要调的工具 / KB
- `observation` —— 工具返回 / KB 返回 / 最终回答

一个最常见的 debug 动作：**把 rationale 和 invocationInput 按顺序打出来**，你立刻能看出模型在哪一步跑偏。

In [ ]:
def print_reasoning(buckets):
    for i, step in enumerate(buckets["orchestration"]):
        if "rationale" in step:
            print(f"[step {i}] THINK: {step['rationale'].get('text','')[:200]}")
        if "invocationInput" in step:
            inp = step["invocationInput"]
            # actionGroupInvocationInput / knowledgeBaseLookupInput / ...
            kind = next((k for k in inp if k != "traceId"), "?")
            print(f"[step {i}] CALL:  {kind} -> {json.dumps(inp.get(kind), default=str)[:200]}")
        if "observation" in step:
            obs = step["observation"]
            print(f"[step {i}] OBS:   {json.dumps(obs, default=str)[:200]}")

# print_reasoning(buckets)

## 4. Model Invocation Logging：看模型真正吃到什么

Trace 已经包含 `modelInvocationInput` 了，但有两个限制：

- 只在 invoke_agent 的 event stream 里存在（客户端不保存 = 丢）
- 不覆盖直接的 `bedrock-runtime` 调用（第 12 课手写 agent）

**Model Invocation Logging** 是账户级配置：打开后 AWS 把"任何对 Bedrock 模型的调用"（包括 agent 内部的）都**自动**写到 CloudWatch Logs 或 S3。

打开方式（一个账户一个 region 配一次即可）：

In [ ]:
# bedrock.put_model_invocation_logging_configuration(
#     loggingConfig={
#         "cloudWatchConfig": {
#             "logGroupName": "/aws/bedrock/modelinvocations",
#             "roleArn":      "arn:aws:iam::...:role/BedrockLoggingRole",
#         },
#         # 也可以 s3Config；两者可以同时开
#         "textDataDeliveryEnabled":      True,
#         "imageDataDeliveryEnabled":     False,
#         "embeddingDataDeliveryEnabled": False,
#     }
# )
print("(配置示例)")

开了之后，**每一次 Bedrock 模型调用**都在 `/aws/bedrock/modelinvocations` log group 里留一条。

**敏感提醒**：这会把 prompt 和 response **原文**写进 log —— 生产上要考虑：
- 权限：谁能读这个 log group
- 保留期：默认永久，建议设 30/90 天
- 合规：如果 prompt 里可能有 PII，要么别开、要么加 KMS、要么脱敏后再进模型

## 5. 四个必盯的 Metrics

CloudWatch 自动发出的 Bedrock Agent 指标（namespace: `AWS/Bedrock`）里，**这四个盯不住线上必出事**：

| 指标 | 为什么重要 | 告警阈值（起点） |
|---|---|---|
| **Invocations** | 调用量；突增可能是被滥用或死循环 | 同比 +200% |
| **InvocationLatency** | p99 延迟；agent 的长尾很吓人 | p99 > 30s |
| **InvocationClientErrors / ServerErrors** | 失败率 | > 1% |
| **InputTokenCount + OutputTokenCount** | 成本 | 单次 > 50k 或日总 > 预算 70% |

另外业务层要自己打的（EMF / put_metric_data）：
- **成功率**（答了 vs 拒绝 / 超时）
- **步数分布**（多少比例的请求走满了 max_steps —— 这是循环征兆）
- **工具调用分布**（哪个工具被调最多，单工具调用均次）

### 发自定义指标的一个最小例子

In [ ]:
def record_agent_run(success: bool, steps: int, tool_calls: int):
    cw.put_metric_data(
        Namespace="AgentCourse/Demo",
        MetricData=[
            {"MetricName": "Success",     "Value": 1 if success else 0, "Unit": "Count"},
            {"MetricName": "StepCount",   "Value": steps,                "Unit": "Count"},
            {"MetricName": "ToolCalls",   "Value": tool_calls,           "Unit": "Count"},
        ],
    )

# 从 trace buckets 提特征然后打点
# record_agent_run(success=True, steps=len(buckets["orchestration"]), tool_calls=3)

## 6. X-Ray：跨服务链路

Agent → Action Group Lambda → 你的 DynamoDB / 外部 API。只看 Bedrock 日志你看不出到底是 Lambda 慢还是 Dynamo 慢。

**开法**：
- Lambda 配置里开 `Active tracing`
- Lambda 代码里用 `aws_xray_sdk` 包一层你的关键调用
- 给 agent role + lambda role 加 `AWSXRayDaemonWriteAccess`

X-Ray 控制台里能看到一条完整 trace：invoke_agent 耗时 X，里面 Lambda 调用耗时 Y，Lambda 里 Dynamo 耗时 Z。

生产排查性能问题基本离不开这个。

## 7. 排查顺序（cheat sheet）

线上某个 agent 调用挂了，按这个顺序看：

1. **Metrics 大盘**：是个例还是全挂？是延迟爆炸还是错误率飙？
2. **找到那次 invocation**：有 request_id / sessionId 就直接搜 log；没有就按时间窗口找
3. **Trace（那次的）**：
   - 有 `failure` trace 吗？看 reason
   - `orchestration` 走了几步？最后一步是 text 还是工具？
   - 中间哪一步 observation 里有 error？
4. **如果是 Lambda 挂**：去 `/aws/lambda/<name>` 的 log group，按 sessionId 搜
5. **如果是模型说错话**：去 `/aws/bedrock/modelinvocations`，看那一步的 prompt 到底长什么样
6. **Guardrail 拦了**：trace 里 `guardrail` bucket 会有 assessment（第 16 课）

**一个硬要求**：业务代码里**每次 invoke_agent 前 log 一条 `{request_id, user_id, sessionId}`**。没有这行，上面 6 步的第 2 步你就断了。

## 8. 脱离 AWS：OpenTelemetry 的位置

团队通常已经有一套 OTEL / Datadog / Grafana。整合路径：

- **CloudWatch metrics → Datadog/Grafana**：走官方 forwarder
- **X-Ray → OTEL**：AWS Distro for OpenTelemetry（ADOT）能把 X-Ray trace 转 OTLP 发出去
- **业务代码层自己打 span**：在 invoke_agent 外面包一个 OTEL span，attrs 里塞 agentId/aliasId/sessionId/tokenCount

Bedrock 原生不吐 OTLP —— 要么接 forwarder，要么靠自己包装。

## 9. 工程直觉

1. **trace 默认开**。生产别关，关了就是在事故里裸奔。
2. **必有 request_id**：贯穿 agent → lambda → 你的 DB 日志 → 前端。没有它，分布式 debug 就等于盲拼。
3. **成功率 + 步数分布 + 延迟 p99**：这三个指标掉一次就该有人被叫醒。
4. **Model Invocation Logging** 是"最后一道真相"：trace 可能丢，metrics 会聚合，只有这里是原始 prompt/response。敏感性高，权限收紧。
5. **别把 debug 逻辑写在 prompt 里**。很多人试图让模型解释自己决策 —— 贵、不准、占 token。要可观测就从系统外面观测。
6. **告警分层**：metrics 层触发值班通知；日志只在 debug 时用。两者混一起的告警系统没人看。

---

下一课：**Multi-agent on Bedrock** —— 单 agent 看得见了，下一题是怎么让多个 agent 在 Bedrock 上协作。Supervisor/Collaborator 模式，对应 Phase 1 第 8 课的托管版。